# Bert Clasificacion Consulta Completa

Notebook que entrena el modelo de clasificacion BERT para clasificar si hay suficiente informacion

## Imports

In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

# Configuración del dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cargar los datos
# Asegúrate de ajustar la ruta al archivo de tus datos
data = pd.read_csv('ruta_a_tus_datos.csv')

# Mostrar un vistazo a los datos
print(data.head())

ModuleNotFoundError: No module named 'torch'

## Preprocesar datos

In [ ]:
# Preprocesar los datos
# Nos enfocamos en las columnas 'Pregunta' y 'Consulta completa'
data = data[['Pregunta', 'Consulta completa']]
data['Consulta completa'] = data['Consulta completa'].astype(int)  # Convertimos el booleano a entero (0 o 1)

# Dividir los datos en conjunto de entrenamiento y prueba
train_texts, test_texts, train_labels, test_labels = train_test_split(data['Pregunta'], data['Consulta completa'], test_size=0.2)

# Inicializar el tokenizador de BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenizar las preguntas
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=128)

# Convertir las etiquetas a tensores
train_labels = torch.tensor(train_labels.values)
test_labels = torch.tensor(test_labels.values)

# Crear tensores para los datos tokenizados
train_dataset = TensorDataset(torch.tensor(train_encodings['input_ids']), torch.tensor(train_encodings['attention_mask']), train_labels)
test_dataset = TensorDataset(torch.tensor(test_encodings['input_ids']), torch.tensor(test_encodings['attention_mask']), test_labels)

# Crear DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Cargar el modelo pre-entrenado BERT
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.to(device)

model

In [ ]:
# Configurar el optimizador
optimizer = AdamW(model.parameters(), lr=1e-5)

## Entrenamiento

In [ ]:
# Entrenamiento del modelo
epochs = 3
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader):
        optimizer.zero_grad()
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f'Epoch {epoch + 1}, Loss: {total_loss / len(train_loader)}')

## Evaluacion de Entrenamiento

In [ ]:
# Evaluación del modelo en el conjunto de prueba
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=-1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f'Accuracy: {accuracy * 100:.2f}%')